<a href="https://colab.research.google.com/github/rajasekhar1103/house-price-prediction/blob/main/reddit_sarcasm_qwen3_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reddit Sarcasm LoRA fine-tuning with Qwen3-4B

This notebook downloads the Reddit Sarcasm dataset and fine-tunes Qwen3-4B with a LoRA adapter. It supports both Kaggle and Colab. In Kaggle, enable **Internet** in the notebook settings and select a GPU accelerator. Colab uses **Runtime → Change runtime type → T4 GPU** or another CUDA GPU. Runtime files and caches are temporary and are removed when the session resets.

Run the cells in order. The default limits training to 50,000 examples and 2,000 validation examples.

In [1]:
%pip -q install --upgrade huggingface_hub datasets trl unsloth

## Download the Reddit Sarcasm dataset

In Kaggle the file is saved under `/kaggle/working/reddit-sarcasm`; in Colab it is saved under `/content/reddit-sarcasm`.

In [2]:
from pathlib import Path
from huggingface_hub import hf_hub_download

runtime_dir = "/kaggle/working" if Path("/kaggle/working").exists() else "/content"
dataset_dir = f"{runtime_dir}/reddit-sarcasm"
hf_cache_dir = f"{runtime_dir}/huggingface_cache"

dataset_file = hf_hub_download(
    repo_id="Thewillonline/reddit-sarcasm",
    filename="reddit-sarcasm.txt",
    repo_type="dataset",
    local_dir=dataset_dir,
)

file_size_gb = Path(dataset_file).stat().st_size / 1_000_000_000
print(f"Dataset downloaded to: {dataset_file}")
print(f"Dataset size: {file_size_gb:.2f} GB")

reddit-sarcasm.txt: reconstructing file:   0%|          |  0.00B /  240MB            

reddit-sarcasm.txt: downloading bytes:           |  0.00B            

Dataset downloaded to: /content/reddit-sarcasm/reddit-sarcasm.txt
Dataset size: 0.24 GB


## Prepare training examples

The file alternates `User:` and `Reddit Comment:` sections. This cell parses 50,000 training and 2,000 validation examples, then creates a conversational text field.

In [ ]:
import re
from datasets import Dataset

MAX_TRAIN = 50_000
MAX_VAL = 2_000
MAX_CHARS = 1_500

pattern = re.compile(r"\bUser:\s*(.*?)\s*Reddit Comment:\s*(.*?)\s*(?:<\|endoftext\|>|\Z)", re.DOTALL)
examples = []

with open(dataset_file, "r", encoding="utf-8", errors="replace") as source:
    text = source.read()

for match in pattern.finditer(text):
    prompt = re.sub(r"\s+", " ", match.group(1)).strip()
    response = re.sub(r"\s+", " ", match.group(2)).strip()
    if 8 <= len(prompt) <= MAX_CHARS and 3 <= len(response) <= MAX_CHARS:
        examples.append({"prompt": prompt, "response": response})
        if len(examples) >= MAX_TRAIN + MAX_VAL:
            break

if len(examples) < MAX_VAL:
    raise RuntimeError("Not enough usable examples were parsed from the dataset")

dataset = Dataset.from_list(examples).train_test_split(test_size=MAX_VAL, seed=42, shuffle=True)
train_dataset = dataset["train"]
val_dataset = dataset["test"]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print(train_dataset[0])

## Load Qwen3-4B and attach LoRA

The model cache is stored under `huggingface_cache` in the active runtime directory. A GPU is required.

In [3]:
import os
import torch
from unsloth import FastLanguageModel

os.environ["HF_HOME"] = hf_cache_dir

if not torch.cuda.is_available():
    raise RuntimeError("A GPU is required. In Colab, select Runtime > Change runtime type > T4 GPU, then rerun this cell.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("Qwen3-4B and LoRA are ready")

W1123 11:34:05.273000 3354 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1123 11:34:05.328000 3354 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth 2026.11.9: no GPU found, training on CPU (x86_64, 1 threads, bfloat16 = False). CPU training is much slower than GPU training; supported: text LoRA / full finetuning with SFTTrainer. Set UNSLOTH_DISABLE_CPU_FALLBACK=1 to raise instead.


RuntimeError: A GPU is required. In Colab, select Runtime > Change runtime type > T4 GPU, then rerun this cell.

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for prompt, response in zip(examples["prompt"], examples["response"]):
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response},
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)
print(train_dataset[0]["text"][:500])

## Train with supervised fine-tuning

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        output_dir=f"{runtime_dir}/qwen3-sarcasm-lora",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

## Save and test the LoRA adapter

In [ ]:
FastLanguageModel.for_inference(model)
adapter_dir = f"{runtime_dir}/qwen3-sarcasm-lora/final_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

messages = [{"role": "user", "content": "This is a great Monday morning and I am excited to start work."}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(inputs, max_new_tokens=120, temperature=0.7, top_p=0.9, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))